In [1]:
print("Hello")

Hello


In [2]:
import os
import glob
import re
import numpy as np
import xarray as xr
import rioxarray  # ensures rasterio backend for xarray
from pathlib import Path

# Define directories
india_dir = Path('India')
nepal_dir = Path('Nepal')

# Get sorted file lists
india_files = sorted(india_dir.glob('ind_pd_*_1km.tif'), key=lambda p: int(re.search(r'(\d{4})', p.name)[1]))
nepal_files = sorted(nepal_dir.glob('npl_pd_*_1km.tif'), key=lambda p: int(re.search(r'(\d{4})', p.name)[1]))

years = range(2000, 2014)
time = xr.Variable('time', [f'{y}-01-01' for y in years])  # arbitrary day[web:1][web:16]

combined_das = []
for y in years:
    # Load India and Nepal
    india_path = india_dir / f'ind_pd_{y}_1km.tif'
    nepal_path = nepal_dir / f'npl_pd_{y}_1km.tif'
    
    da_ind = rioxarray.open_rasterio(str(india_path)).squeeze('band', drop=True)
    da_npl = rioxarray.open_rasterio(str(nepal_path)).squeeze('band', drop=True)
    
    # Simple mosaic: max to handle no-data (assuming nodata where 0 or NaN)
    da_year = xr.where(da_ind.notnull(), da_ind, da_npl).max(dim=None)  # or .fillna(0).sum() if no overlap
    combined_das.append(da_year)

# Stack along time
ds = xr.concat(combined_das, dim=time).to_dataset(name='population_density')

# Save as NetCDF (preserves CRS, coords)
ds.to_netcdf('indo_nepal_pd_2000_2013.nc')
print('Saved indo_nepal_pd_2000_2013.nc')

AlignmentError: cannot align objects with join='exact' where index/labels/sizes are not equal along these coordinates (dimensions): 'x' ('x',)